# House Price Prediction
## Objective
The goal of this notebook is to build a robust regression model to predict house prices based on features like BHK, Type, Area, and Status. We will compare multiple regression algorithms and select the best one for deployment.

### Step 1: Data Loading
**What**: Loading the raw house price dataset from the CSV file.
**Why**: To start our analysis, we first need to bring the data into our Python environment.
**How**: Using the `read_csv` function from the `pandas` library.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

df = pd.read_csv('house_prices_new_data.csv')
df.head()

,BHK,Type,Area,Status,Price,price_unit
0,2,Apartment,605,Under Construction,3780000,L
1,1,Apartment,557,Under Construction,2378250,L
2,2,Apartment,635,Under Construction,3780000,L
3,1,Apartment,500,Under Construction,2250000,L
4,3,Apartment,1259,Ready to move,6536000,L


### Step 2: Exploratory Data Analysis (EDA)
**What**: Inspecting the structure and basic statistics of the data.
**Why**: Understanding the data types, checking for missing values, and exploring unique values in categorical columns helps us decide which preprocessing steps are needed.
**How**: Using `df.info()` for structural overview and `unique()` to check categorical levels.

In [ ]:
df.info()
print('\nUnique values in Status:', df['Status'].unique())
print('Unique values in Type:', df['Type'].unique())

### Step 3: Data Cleaning
**What**: Standardizing textual data and formatting numerical values.
**Why**: Machine learning models are sensitive to formatting inconsistencies (e.g., 'Ready to move' vs 'Ready to Move'). Numerical columns like 'Price' must be of type float for calculations.
**How**: Using string methods like `.strip()` and `.capitalize()` and removing commas from 'Price' using `.str.replace()`.

In [ ]:
# Normalize Status and Type strings
df['Status'] = df['Status'].str.strip().str.capitalize()
df['Type'] = df['Type'].str.strip().str.capitalize()

# Clean Price column: Remove commas and convert to float
if df['Price'].dtype == 'O':
    df['Price'] = df['Price'].str.replace(',', '').astype(float)

df.head()

### Step 4: Data Splitting
**What**: Dividing the dataset into training and testing sets.
**Why**: To evaluate our model's performance on unseen data (the test set) to ensure it generalizes well and doesn't just memorize the training data (overfitting).
**How**: Using `train_test_split` with 80% for training and 20% for testing.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[['BHK', 'Type', 'Area', 'Status']]
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Step 5: Defining Preprocessing Steps
**What**: Setting up a `ColumnTransformer` for categorical and numerical features.
**Why**: Models require numerical input. Categorical columns (Type, Status) need One-Hot Encoding, while numerical columns (BHK, Area) can pass through.
**How**: Using `OneHotEncoder` for categories and 'passthrough' for numbers.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_features = ['Type', 'Status']
numeric_features = ['BHK', 'Area']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numeric_features)
    ]
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_model(y_test, y_pred):
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    return r2, mae, rmse

## Step 6: Individual Model Training and Evaluation

### 6.1 Linear Regression
**What**: Training a simple Linear Regression model.
**Why**: It serves as a baseline model to see how a simple linear relationship performs.
**How**: Creating a pipeline and fitting the training data.

In [ ]:
from sklearn.linear_model import LinearRegression

lr_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', LinearRegression())])
lr_pipeline.fit(X_train, y_train)
lr_pred = lr_pipeline.predict(X_test)

lr_r2, lr_mae, lr_rmse = evaluate_model(y_test, lr_pred)
print(f'Linear Regression - R2 Score: {lr_r2:.4f}, MAE: {lr_mae:.2f}, RMSE: {lr_rmse:.2f}')

**Score Meaning (Linear Regression)**:
- **R2 Score**: How well the model fits the data. 0.98 means 98% of the variance is captured.
- **MAE**: On average, the prediction is off by this amount in Rupees.
- **RMSE**: Penalizes larger errors more heavily. Lower is better.

### 6.2 Random Forest Regressor
**What**: Training an Ensemble Random Forest model.
**Why**: It combines multiple decision trees to improve accuracy and reduce overfitting.
**How**: Fitting 100 decision trees to the training data.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))])
rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)

rf_r2, rf_mae, rf_rmse = evaluate_model(y_test, rf_pred)
print(f'Random Forest - R2 Score: {rf_r2:.4f}, MAE: {rf_mae:.2f}, RMSE: {rf_rmse:.2f}')

**Score Meaning (Random Forest)**:
- **R2 Score**: Typically much higher than Linear Regression as it captures non-linear patterns.
- **MAE**: Should be lower if the model is more accurate in predicting average prices.

### 6.3 XGBoost Regressor
**What**: Training an Extreme Gradient Boosting model.
**Why**: It is one of the most powerful algorithms for structured data, often giving the best accuracy.
**How**: Using the `XGBRegressor` within our pipeline.

In [ ]:
from xgboost import XGBRegressor

xgb_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42))])
xgb_pipeline.fit(X_train, y_train)
xgb_pred = xgb_pipeline.predict(X_test)

xgb_r2, xgb_mae, xgb_rmse = evaluate_model(y_test, xgb_pred)
print(f'XGBoost - R2 Score: {xgb_r2:.4f}, MAE: {xgb_mae:.2f}, RMSE: {xgb_rmse:.2f}')

**Score Meaning (XGBoost)**:
- **R2 Score**: A high score (e.g. 0.99) indicates near-perfect prediction capability.
- **RMSE**: A very low RMSE means the model is extremely confident across all price ranges.

### Step 7: Final Model Comparison and Selection
**What**: Comparing the performance of all three models side-by-side.
**Why**: To objectively select the most accurate model for deployment.
**How**: Summarizing results in a table and saving the best one.

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'XGBoost'],
    'R2 Score': [lr_r2, rf_r2, xgb_r2],
    'MAE': [lr_mae, rf_mae, xgb_mae],
    'RMSE': [lr_rmse, rf_rmse, xgb_rmse]
})

print(results)

# Logic to pick the best model for saving
best_model = rf_pipeline if rf_r2 > xgb_r2 else xgb_pipeline
best_model_name = 'Random Forest' if rf_r2 > xgb_r2 else 'XGBoost'
print(f'\nSelected the {best_model_name} as the final model.')

### Step 8: Saving the Artifacts
**What**: Saving the selected model pipeline to a file.
**Why**: So the Streamlit app can load it easily without needing the training code.
**How**: Using `joblib.dump()` to save to the `artifacts` directory.

In [ ]:
if not os.path.exists('artifacts'):
    os.makedirs('artifacts')

joblib.dump(best_model, 'artifacts/house_price_model.pkl')
print(f'Successfully saved the {best_model_name} model.')